#### Geospatial/Amenity based Analytical Queries

In [0]:
%sql
-- ── G1. Sites within 300m of a hotel ─────────────────────
-- Direct proximity filter using pre-computed relationship_type
-- No spatial math at query time — pure filter on stored band
SELECT
    dp.property_id,
    dp.boro,
    dp.street_name,
    dp.zip_code,
    dp.zoning,
    fsa.distance_meters,
    dh.owner_name                           AS hotel_name,
    dh.street_name                          AS hotel_street
FROM cre_catalog.gold.fact_site_amenity fsa
JOIN cre_catalog.gold.dim_property dp
    ON fsa.site_id = dp.property_id
JOIN cre_catalog.gold.dim_hotel dh
    ON fsa.amenity_id = dh.hotel_id
WHERE fsa.amenity_type = 'hotel'
  AND fsa.relationship_type IN ('within_100m', 'within_300m')
  AND dp.is_current = TRUE
ORDER BY fsa.distance_meters
LIMIT 50;

In [0]:
%sql
-- ── G2. Rank sites by number of nearby hotels within 1km ──
SELECT
    dp.property_id,
    dp.boro,
    dp.street_name,
    dp.zip_code,
    COUNT(fsa.amenity_id)                   AS hotels_within_1km,
    MIN(fsa.distance_meters)                AS nearest_hotel_m,
    ROUND(AVG(fsa.distance_meters), 0)      AS avg_hotel_distance_m
FROM cre_catalog.gold.fact_site_amenity fsa
JOIN cre_catalog.gold.dim_property dp
    ON fsa.site_id = dp.property_id
WHERE fsa.amenity_type = 'hotel'
  AND dp.is_current = TRUE
GROUP BY dp.property_id, dp.boro, dp.street_name, dp.zip_code
ORDER BY hotels_within_1km DESC
LIMIT 30;

In [0]:
%sql
-- ── G3. Recent sales near hotels (within 300m) ────────────
-- Investment activity happening near hospitality clusters
SELECT
    fs.property_id,
    dp.boro,
    dp.street_name,
    dp.zip_code,
    fs.sale_year,
    fs.seller_name,
    fs.buyer_name,
    fsa.distance_meters                     AS dist_to_hotel_m,
    dh.owner_name                           AS nearby_hotel_owner,
    dh.street_name                          AS hotel_street
FROM cre_catalog.gold.fact_sale fs
JOIN cre_catalog.gold.dim_property dp
    ON fs.property_id = dp.property_id
JOIN cre_catalog.gold.fact_site_amenity fsa
    ON fs.property_id = fsa.site_id
    AND fsa.amenity_type = 'hotel'
    AND fsa.relationship_type = 'within_300m'
JOIN cre_catalog.gold.dim_hotel dh
    ON fsa.amenity_id = dh.hotel_id
WHERE fs.sale_year >= 2020
ORDER BY fs.sale_year DESC, fsa.distance_meters
LIMIT 30;

In [0]:
%sql
-- ── G4. Amenity density heatmap by H3 cell ────────────────
-- Groups sites and amenities by shared H3 index
-- Reveals which hexagonal zones have highest amenity concentration
-- Can be visualized directly on a map using H3 cell coordinates
SELECT
    dp.h3_index,
    dp.boro,
    COUNT(DISTINCT dp.property_id)          AS num_sites,
    COUNT(DISTINCT CASE WHEN fsa.amenity_type = 'restaurant'
          THEN fsa.amenity_id END)          AS num_restaurants,
    COUNT(DISTINCT CASE WHEN fsa.amenity_type = 'hotel'
          THEN fsa.amenity_id END)          AS num_hotels,
    COUNT(DISTINCT fsa.amenity_id)          AS total_amenities,
    ROUND(AVG(fsa.distance_meters), 0)      AS avg_proximity_m
FROM cre_catalog.gold.dim_property dp
LEFT JOIN cre_catalog.gold.fact_site_amenity fsa
    ON dp.property_id = fsa.site_id
WHERE dp.is_current = TRUE
  AND dp.h3_index IS NOT NULL
GROUP BY dp.h3_index, dp.boro
ORDER BY total_amenities DESC
LIMIT 50;

In [0]:
%sql

-- ── G5. Walkability score per site ────────────────────────
-- Composite score based on number and proximity of amenities
-- Closer amenities score higher — mirrors Walk Score methodology
SELECT
    dp.property_id,
    dp.boro,
    dp.street_name,
    dp.zip_code,
    dp.zoning,
    COUNT(CASE WHEN fsa.relationship_type = 'within_100m' THEN 1 END) AS amenities_100m,
    COUNT(CASE WHEN fsa.relationship_type = 'within_300m' THEN 1 END) AS amenities_300m,
    COUNT(CASE WHEN fsa.relationship_type = 'within_500m' THEN 1 END) AS amenities_500m,
    COUNT(CASE WHEN fsa.relationship_type = 'within_1km'  THEN 1 END) AS amenities_1km,
    -- Weighted score: closer = more weight
    ROUND(
        COUNT(CASE WHEN fsa.relationship_type = 'within_100m' THEN 1 END) * 4 +
        COUNT(CASE WHEN fsa.relationship_type = 'within_300m' THEN 1 END) * 3 +
        COUNT(CASE WHEN fsa.relationship_type = 'within_500m' THEN 1 END) * 2 +
        COUNT(CASE WHEN fsa.relationship_type = 'within_1km'  THEN 1 END) * 1
    , 0)                                    AS walkability_score
FROM cre_catalog.gold.dim_property dp
JOIN cre_catalog.gold.fact_site_amenity fsa
    ON dp.property_id = fsa.site_id
WHERE dp.is_current = TRUE
GROUP BY dp.property_id, dp.boro, dp.street_name, dp.zip_code, dp.zoning
ORDER BY walkability_score DESC
LIMIT 50;


In [0]:
%sql
-- ── G6. Same H3 cell co-location ─────────────────────────
-- Sites that share an H3 cell with an amenity (< ~174m)
-- h3_same_cell = true means they are in the exact same hexagon
-- Most precise proximity indicator without exact distance
SELECT
    dp.property_id,
    dp.boro,
    dp.street_name,
    fsa.amenity_type,
    fsa.distance_meters,
    CASE WHEN fsa.amenity_type = 'hotel'
         THEN dh.owner_name
         ELSE dr.name
    END                                     AS amenity_name,
    CASE WHEN fsa.amenity_type = 'hotel'
         THEN dh.street_name
         ELSE dr.street_name
    END                                     AS amenity_street
FROM cre_catalog.gold.fact_site_amenity fsa
JOIN cre_catalog.gold.dim_property dp
    ON fsa.site_id = dp.property_id
LEFT JOIN cre_catalog.gold.dim_hotel dh
    ON fsa.amenity_id = dh.hotel_id
    AND fsa.amenity_type = 'hotel'
LEFT JOIN cre_catalog.gold.dim_restaurant dr
    ON fsa.amenity_id = dr.restaurant_id
    AND fsa.amenity_type = 'restaurant'
WHERE fsa.h3_same_cell = TRUE
  AND dp.is_current = TRUE
ORDER BY fsa.distance_meters
LIMIT 50;

In [0]:
%sql
-- ── G7. High value parcels near restaurants ───────────────
-- Site selection: premium assets with strong food amenity access
SELECT
    dp.property_id,
    dp.boro,
    dp.street_name,
    dp.zip_code,
    fta.cur_act_total                       AS assessed_value,
    fta.gross_sqft,
    ROUND(fta.cur_act_total
        / NULLIF(fta.gross_sqft, 0), 2)     AS av_per_sqft,
    COUNT(fsa.amenity_id)                   AS nearby_restaurants,
    MIN(fsa.distance_meters)                AS nearest_restaurant_m
FROM cre_catalog.gold.fact_tax_assessment fta
JOIN cre_catalog.gold.dim_property dp
    ON fta.site_id = dp.property_id
JOIN cre_catalog.gold.fact_site_amenity fsa
    ON dp.property_id = fsa.site_id
    AND fsa.amenity_type = 'restaurant'
    AND fsa.relationship_type IN ('within_100m', 'within_300m')
WHERE fta.tax_year = (SELECT MAX(tax_year) FROM cre_catalog.gold.fact_tax_assessment)
  AND fta.cur_act_total > 10000000
  AND dp.is_current = TRUE
GROUP BY dp.property_id, dp.boro, dp.street_name,
         dp.zip_code, fta.cur_act_total, fta.gross_sqft
ORDER BY av_per_sqft DESC
LIMIT 50;

#### Property based Analytics

In [0]:
%sql
-- ── 1. Portfolio overview: total assessed value by borough ─
-- Shows the tax base distribution across NYC boroughs
SELECT
    dp.boro,
    COUNT(DISTINCT fta.site_id)             AS num_parcels,
    ROUND(SUM(fta.cur_act_total), 0)        AS total_assessed_value,
    ROUND(AVG(fta.cur_act_total), 0)        AS avg_assessed_value,
    ROUND(SUM(fta.cur_txb_total), 0)        AS total_taxable_value
FROM cre_catalog.gold.fact_tax_assessment fta
JOIN cre_catalog.gold.dim_property dp
    ON fta.site_id = dp.property_id
WHERE fta.tax_year = (SELECT MAX(tax_year) FROM cre_catalog.gold.fact_tax_assessment)
  AND dp.is_current = TRUE
GROUP BY dp.boro
ORDER BY total_assessed_value DESC;

In [0]:
%sql
-- ── 2. Year-over-year assessed value change ───────────────
-- Tracks AV movement from prior year to current per parcel
SELECT
    fta.site_id,
    dp.boro,
    dp.street_name,
    dp.zip_code,
    fta.tax_year,
    fta.py_act_total                                        AS av_prior_year,
    fta.cur_act_total                                       AS av_current,
    ROUND(fta.cur_act_total - fta.py_act_total, 0)         AS av_change,
    ROUND(
        (fta.cur_act_total - fta.py_act_total)
        / NULLIF(fta.py_act_total, 0) * 100, 2
    )                                                       AS pct_change
FROM cre_catalog.gold.fact_tax_assessment fta
JOIN cre_catalog.gold.dim_property dp
    ON fta.site_id = dp.property_id
WHERE fta.tax_year = (SELECT MAX(tax_year) FROM cre_catalog.gold.fact_tax_assessment)
  AND fta.py_act_total > 0
  AND dp.is_current = TRUE
ORDER BY pct_change DESC
LIMIT 50;

In [0]:
%sql
-- ── 3. Top owners by number of properties ─────────────────
-- Identifies major landlords and institutional holders
SELECT
    do.owner_name,
    COUNT(DISTINCT fta.site_id)             AS num_parcels,
    ROUND(SUM(fta.cur_act_total), 0)        AS total_portfolio_av,
    ROUND(SUM(fta.gross_sqft), 0)           AS total_gross_sqft,
    COUNT(DISTINCT dp.boro)                 AS boroughs_active
FROM cre_catalog.gold.fact_tax_assessment fta
JOIN cre_catalog.gold.dim_owner do
    ON fta.owner_id = do.owner_id
JOIN cre_catalog.gold.dim_property dp
    ON fta.site_id = dp.property_id
WHERE fta.tax_year = (SELECT MAX(tax_year) FROM cre_catalog.gold.fact_tax_assessment)
  AND do.owner_name NOT IN ('CITY OF NEW YORK', 'NYC HOUSING AUTHORITY', 'NYC DEPT OF CITYWIDE ADMIN SVCS')
GROUP BY do.owner_name
ORDER BY num_parcels DESC
LIMIT 25;